# Estimation des Stocks de Carbone et Biomasse

Ce notebook utilise les indices de végétation pour estimer la biomasse aérienne et le carbone stocké dans la zone d'étude.

In [ ]:
!pip install geemap earthengine-api rasterio matplotlib -q
import ee, geemap, rasterio
import numpy as np
import matplotlib.pyplot as plt

ee.Initialize(project='geocongoai-api')

In [ ]:
roi = ee.Geometry.Rectangle([15.0, -5.0, 15.4, -4.6])
image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).median().clip(roi)
geemap.ee_export_image(image.select(['B4', 'B8']), 'input.tif', scale=30, region=roi)

In [ ]:
with rasterio.open('input.tif') as src: data = src.read().astype(np.float32)
red, nir = data[0], data[1]
ndvi = (nir - red) / (nir + red + 1e-8)

biomass = 150 * np.maximum(0, ndvi)**2
carbon = 0.5 * biomass

plt.imshow(carbon, cmap='YlGn')
plt.colorbar(label='Carbone estimé (t/ha)')
plt.title("Estimation des Stocks de Carbone Forestier")
plt.show()